<a href="https://colab.research.google.com/github/georgmandryka-cyber/lager-chatbot-rag/blob/main/lager_chatbot_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install -q anthropic sentence-transformers

import anthropic
from pathlib import Path
from google.colab import userdata
from sentence_transformers import SentenceTransformer, util

# 1. Beispiel-Dateien anlegen (nur wenn der Ordner noch fehlt)
ordner = Path("unterlagen")
if not ordner.exists():
    ordner.mkdir()
    beispiele = {
        "gefahrgut.txt": [
            "Gefahrgut wird nur im gekennzeichneten Bereich C gelagert.",
            "Entzündbare Stoffe dürfen nicht über 2 Paletten hoch gestapelt werden.",
            "Bei einem Leck wird sofort der Bereich abgesperrt und der Schichtleiter informiert.",
        ],
        "stapler.txt": [
            "Vor jeder Schicht prüft der Fahrer Gabeln, Bremsen und Hupe.",
            "Mängel werden sofort dem Schichtleiter gemeldet, der Stapler wird gesperrt.",
        ],
        "wareneingang.txt": [
            "Bei Abweichungen von mehr als 2 Prozent wird der Schichtleiter informiert.",
        ],
        "verkehr.txt": [
            "Die Höchstgeschwindigkeit in der Halle beträgt 10 km/h.",
        ],
        "pausen.txt": [
            "Die Frühstückspause dauert 15 Minuten.",
            "Die Mittagspause dauert 30 Minuten.",
        ],
    }
    for name, zeilen in beispiele.items():
        (ordner / name).write_text("\n".join(zeilen), encoding="utf-8")

# 2. Alle .txt-Dateien einlesen: eine Zeile = ein Textstück
chunks = []
for datei in sorted(ordner.glob("*.txt")):
    for zeile in datei.read_text(encoding="utf-8").splitlines():
        if zeile.strip():
            chunks.append({"thema": datei.stem.capitalize(),
                           "quelle": datei.name,
                           "text": zeile.strip()})
print(f"{len(chunks)} Textstücke aus {len(list(ordner.glob('*.txt')))} Dateien geladen")

# 3. Retrieval mit Embeddings (Thema + Text)
modell = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
chunk_vektoren = modell.encode([c["thema"] + ": " + c["text"] for c in chunks])
SCHWELLE = 0.35

def suche(frage, n=3):
    frage_vektor = modell.encode(frage)
    werte = util.cos_sim(frage_vektor, chunk_vektoren)[0].tolist()
    treffer = sorted(zip(werte, chunks), key=lambda t: t[0], reverse=True)
    return treffer[:n]

# 4. Generation mit Claude
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

SYSTEM = (
    "Du bist ein Assistent für Lager-Arbeitsanweisungen. "
    "Antworte kurz und auf Deutsch, nur anhand der gegebenen Auszüge. "
    "Steht die Antwort nicht in den Auszügen, sag das ehrlich. "
    "Nenne am Ende die Quelle."
)

def antworten(frage):
    treffer = suche(frage)
    if treffer[0][0] < SCHWELLE:
        return "Dazu habe ich nichts in den Unterlagen gefunden."
    kontext = "\n".join(f"- ({c['quelle']}) {c['text']}" for _, c in treffer)
    antwort = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        system=SYSTEM,
        messages=[{"role": "user",
                   "content": f"Auszüge:\n{kontext}\n\nFrage: {frage}"}],
    )
    return antwort.content[0].text

# 5. Test
for frage in ["Wo lagere ich Gefahrgut?",
              "Wie schnell darf der Stapler fahren?",
              "Wie lange darf ich Mittagspause machen?",
              "Wer ist der Bundeskanzler?"]:
    print("FRAGE:", frage)
    print(antworten(frage))
    print()

9 Textstücke aus 5 Dateien geladen


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

FRAGE: Wo lagere ich Gefahrgut?
Gefahrgut wird ausschließlich im **gekennzeichneten Bereich C** gelagert.

*Quelle: gefahrgut.txt*

FRAGE: Wie schnell darf der Stapler fahren?
In der Halle darf der Stapler **maximal 10 km/h** fahren.

*Quelle: verkehr.txt*

FRAGE: Wie lange darf ich Mittagspause machen?
Die Mittagspause dauert **30 Minuten**.

Quelle: pausen.txt

FRAGE: Wer ist der Bundeskanzler?
Dazu habe ich nichts in den Unterlagen gefunden.



In [19]:
print(antworten("Wie lange darf ich Mittagspause machen?"))

Die Mittagspause dauert **30 Minuten**.

Quelle: pausen.txt
